# Swin Transformer v2 画像・タグ学習システム (Google Colab版)

Google Drive 内の **Cryptomator 暗号化保管庫 (Vault)** から **WebDAV** を介してインメモリでストリーミング学習を行います。

> **💡 コスト節約のための推奨手順**:
> Google Colab の貴重な GPU 時間を無駄に消費しないため、**まずは通常の「CPU ランタイム」で実行**し、Google Drive のマウントと Cryptomator の復号・WebDAV 疎通が正常に完了することを確認してください。
> 接続確認が取れた後、メニューの「ランタイムのタイプを変更」で **GPU** に切り替えて本格的な学習・推論に進むことができます。

--- 
## 【フェーズ 1: 事前検証 (CPU ランタイムで実行可能)】
Google Drive と Cryptomator の設定が正常に機能するか確認します。

### 1. Google Drive のマウント & Vault フォルダ事前診断

In [4]:
from google.colab import drive
import os

# 1. Google Drive をマウント
drive.mount('/content/drive')

# 2. Google Drive 内の Vault (保管庫) フォルダパスを指定してください
# 例: '/content/drive/MyDrive/MyVault' など
VAULT_PATH = '/content/drive/MyDrive/hentai/hentai'  # ← あなたの保管庫パスに変更してください

# 事前診断チェック
print(f"\n--- 保管庫パスの診断: {VAULT_PATH} ---")
if os.path.exists(VAULT_PATH):
    print("✅ 保管庫フォルダが存在することを確認しました。")
    files = os.listdir(VAULT_PATH)
    has_vault_config = any("vault.cryptomator" in f or "masterkey.cryptomator" in f for f in files)
    if has_vault_config:
        print("✅ Cryptomator 保管庫の設定ファイル (vault.cryptomator) を検出しました。")
    else:
        print("⚠️ 警告: フォルダ内に Cryptomator の設定ファイルが見当たりません。パスが保管庫のルートフォルダを指しているか確認してください。")
else:
    print("❌ エラー: 指定された Vault パスが見つかりません。Google Drive のフォルダ名や大文字小文字を確認してください。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- 保管庫パスの診断: /content/drive/MyDrive/hentai/hentai ---
✅ 保管庫フォルダが存在することを確認しました。
✅ Cryptomator 保管庫の設定ファイル (vault.cryptomator) を検出しました。


### 2. 環境構築 & Cryptomator CLI のセットアップ

In [ ]:
# 1. 必要な Python パッケージのインストール
!pip install -q timm rule34Py python-dotenv pyyaml requests pillow tqdm

# 2. JRE (Java 実行環境) および unzip のセットアップ
!apt-get update -qq && apt-get install -y -qq default-jre unzip > /dev/null

# 3. Cryptomator CLI のインストール (v0.6.2)
!mkdir -p /opt/cryptomator-cli
!wget -q -O /tmp/cryptomator-cli.zip https://github.com/cryptomator/cli/releases/download/0.6.2/cryptomator-cli-0.6.2-linux-x64.zip
!unzip -q -o /tmp/cryptomator-cli.zip -d /opt/cryptomator-cli
!chmod +x /opt/cryptomator-cli/bin/cryptomator-cli
!ln -sf /opt/cryptomator-cli/bin/cryptomator-cli /usr/local/bin/cryptomator-cli
!rm -f /tmp/cryptomator-cli.zip

# インストール確認
!cryptomator-cli --version


### 3. Vault アンロック & WebDAV 疎通テスト（ヘルスチェック）

パスワードを入力して WebDAV を起動し、正しくデータセットが読み取れるか検証します。

In [ ]:
import getpass
import os
import re
import subprocess
import time
import requests
import yaml

if not os.path.exists(VAULT_PATH):
    raise FileNotFoundError(f"Vault パスが存在しません: {VAULT_PATH}")

# パスワード入力 (画面に平文表示されません)
vault_password = getpass.getpass("Cryptomator Vault のパスワードを入力してください: ")

env = os.environ.copy()
env["VAULT_PW"] = vault_password

print("Cryptomator WebDAV サーバーを起動中...")
cmd = [
    "cryptomator-cli",
    "unlock",
    "--mounter", "org.cryptomator.frontend.webdav.mount.FallbackMounter",
    "--password:env=VAULT_PW",
    VAULT_PATH
]

# バックグラウンド起動
crypto_proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

webdav_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = crypto_proc.stdout.readline()
    if line:
        print(line.strip())
        match = re.search(r'(http://(?:localhost|127\.0\.1):\d+/[^\s]+)', line)
        if match:
            webdav_url = match.group(1).rstrip("/")
            break
    if crypto_proc.poll() is not None:
        break
    time.sleep(0.1)

if not webdav_url:
    raise RuntimeError("WebDAV サーバーの起動に失敗しました。パスワードや保管庫パスを確認してください。")

print(f"\n[検知成功] WebDAV URL: {webdav_url}")

# WebDAV 疎通ヘルスチェック (PROPFIND)
try:
    resp = requests.request("PROPFIND", webdav_url, headers={"Depth": "1"}, timeout=10)
    if resp.status_code in (200, 207):
        print("✅ WebDAV サーバーへの HTTP 通信テストに成功しました！")
    else:
        print(f"⚠️ 警告: WebDAV 応答ステータス: {resp.status_code}")
except Exception as e:
    print(f"❌ WebDAV 通信エラー: {e}")

# config.yaml を自動更新
config_path = "config.yaml"
if not os.path.exists(config_path) and os.path.exists("example.config.yaml"):
    with open("example.config.yaml", "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
else:
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}

cfg.setdefault("storage", {})["webdav_url"] = webdav_url
with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
print("✅ config.yaml に接続 URL を反映しました。")

### 4. 保管庫内データセットの読み取り確認

In [ ]:
# 保存済みデータセット一覧を表示
!python main.py list

print("\n" + "=" * 60)
print("🎉 [確認完了] Google Drive と Cryptomator の連携テストがすべて成功しました！")
print("=" * 60)
print("\n【次のステップ】")
print("1. 上部メニューの [ランタイム] -> [ランタイムのタイプを変更] を選択します。")
print("2. ハードウェア アクセラレータで [GPU (T4 など)] を選択して保存します。")
print("3. 下記の【フェーズ 2】へ進み、GPU での高速学習・推論を開始してください！")

--- 
## 【フェーズ 2: GPU での学習 & 推論】
GPU ランタイムに切り替えた後、以下のセルを実行します。

### 5. GPU の認識確認

In [ ]:
!nvidia-smi

### 6. Swin Transformer v2 学習の実行 (GPU 高速化・SSD非保存)

Cryptomator の WebDAV から画像を直接 RAM 上に並列プリロードし、GPU で学習を実行します。
Colab のローカルディスクには平文画像は一切書き込まれません。

In [ ]:
# 省メモリモード (low_vram) または 高精度モード (high_quality) を指定可能
!python main.py train \
    --dataset-name "test_overwatch" \
    --preset high_quality \
    --min-frequency 5 \
    --epochs 20 \
    --batch-size 32

### 7. 学習済みモデルによる推論（タグ予測）

学習したチェックポイント（`.pth`）を使って、画像ファイルまたは Web 上の画像 URL からタグを予測します。

In [ ]:
# テスト画像 URL または Google Drive 内の画像パスを指定
TEST_IMAGE_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"

!python main.py predict \
    --checkpoint "./checkpoints/test_overwatch_best_model.pth" \
    --image "{TEST_IMAGE_URL}" \
    --threshold 0.35 \
    --top-k 15

### 8. クリーンアップ (Cryptomator の安全なロック・終了)

作業終了後、バックグラウンドの WebDAV サーバーを停止し、Vault を安全にロックします。

In [ ]:
try:
    if 'crypto_proc' in globals() and crypto_proc.poll() is None:
        crypto_proc.terminate()
        crypto_proc.wait(timeout=5)
        print("Cryptomator WebDAV サーバーを正常に停止し、Vault をロックしました。")
    else:
        print("Cryptomator プロセスは実行されていません。")
except Exception as e:
    print(f"停止時エラー: {e}")